# Synthetic TC14 — P1 protograph RDF2Vec + MASCHInE finetune

End-to-end notebook for **Synthetic TC14**: build P1/P2 protographs, pretrain Word2Vec on P1 walks (`scripts/_walks.py`), then finetune on instance walks with MASCHInE initialization at the **native pretrained scale** and a reduced finetuning learning rate (same pattern as `dbpedia.ipynb`).

TC14 labels are determined purely by class membership (positives `C_tc14_314`, negatives `C_tc14_435`), and `graph.nt` contains no `rdf:type` edges — so the class signal enters *only* through the protograph init. Two details keep that signal from being washed out during finetuning (see Step 5/6 for the reasoning):

1. **No down-normalization of pretrained rows.** Shrinking protograph vectors (norm ≈ 2.6) to gensim's random-init scale (norm ≈ 0.04) makes every SGNS dot product ≈ 0, so full-error gradients erase the inherited geometry within one epoch (test accuracy 1.00 → 0.65).
2. **Finetuning LR, not from-scratch LR.** `alpha=0.005` instead of `0.025` (1.00 → 0.99 after 5 epochs, vs 1.00 → 0.79 with 0.025).

All artifacts are written under `notebooks/synthetic_tc14/`.

In [17]:
"""Setup paths (conf/p1_default.yaml hyperparameters are inlined at use sites)."""
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"
NOTEBOOKS = ROOT / "notebooks"
OUT_DIR = NOTEBOOKS / "synthetic_tc14"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TC = "tc14"
TC_DIR = ROOT / "v1" / "synthetic_ontology" / TC / "synthetic_ontology"
ONTOLOGY = TC_DIR / "ontology.nt"
GRAPH = TC_DIR / "graph.nt"
TEST_TXT = TC_DIR / "1000" / "train_test" / "test.txt"
TRAIN_TXT = TC_DIR / "1000" / "train_test" / "train.txt"

P1_PATH = OUT_DIR / "protograph_p1.nt"
P2_PATH = OUT_DIR / "protograph_p2.nt"
ENTITY2CLASSES_PATH = OUT_DIR / "entity2classes.json"
P1_WALKS_PATH = OUT_DIR / "walks_p1.txt"
INSTANCE_WALKS_PATH = OUT_DIR / "walks_instance.txt"
PRETRAIN_MODEL_PATH = OUT_DIR / "word2vec_p1.model"
PRETRAIN_KV_PATH = OUT_DIR / "word2vec_p1.kv"
INSTANCE_MODEL_PATH = OUT_DIR / "word2vec_instance.model"
INSTANCE_KV_PATH = OUT_DIR / "word2vec_instance.kv"

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from _walks import nt_term, run_jrdf2vec_duplicate_free  # noqa: E402

for p in (ONTOLOGY, GRAPH, TEST_TXT, TRAIN_TXT):
    assert p.is_file(), f"Missing required input: {p}"

print(f"TC:          {TC}")
print(f"Ontology:    {ONTOLOGY}")
print(f"Instance KG: {GRAPH}")
print(f"Output dir:  {OUT_DIR}")

TC:          tc14
Ontology:    /home/bnukhkadiev/Knowledge-Graphs/v1/synthetic_ontology/tc14/synthetic_ontology/ontology.nt
Instance KG: /home/bnukhkadiev/Knowledge-Graphs/v1/synthetic_ontology/tc14/synthetic_ontology/graph.nt
Output dir:  /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14


In [18]:
"""Step 1: Generate MASCHInE protographs P1 and P2 from the TC14 ontology."""
cmd = [
    sys.executable,
    str(SCRIPTS / "_protograph_gen.py"),
    "--schema", str(ONTOLOGY),
    "--ontology-only",
    "--out-dir", str(OUT_DIR),
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=str(ROOT))

assert P1_PATH.is_file() and P2_PATH.is_file(), "Protograph generation failed"

p1_triples = []
p1_class_tokens: set[str] = set()
p1_prop_tokens: set[str] = set()
with P1_PATH.open(encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        s, p, o = parts[0][1:-1], parts[1][1:-1], parts[2][1:-1]
        p1_triples.append((s, p, o))
        p1_class_tokens.update({s, o})
        p1_prop_tokens.add(p)

with P2_PATH.open(encoding="utf-8") as f:
    n_p2 = sum(1 for line in f if line.strip())

print(f"P1 triples:  {len(p1_triples):,}")
print(f"P2 triples:  {n_p2:,}")
print(f"P1 classes:  {len(p1_class_tokens):,}")
print(f"P1 props:    {len(p1_prop_tokens):,}")
print(f"Saved:       {P1_PATH.name}, {P2_PATH.name}")

$ /home/bnukhkadiev/Knowledge-Graphs/.venv/bin/python /home/bnukhkadiev/Knowledge-Graphs/scripts/_protograph_gen.py --schema /home/bnukhkadiev/Knowledge-Graphs/v1/synthetic_ontology/tc14/synthetic_ontology/ontology.nt --ontology-only --out-dir /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14
P1 sanity check passed (notebook style: one triple per relation).
P2 sanity check passed (each direct subclass of a P1 domain/range class appears in an additional triple, per Sec. 3.1).
Entity-class mapping sanity check passed (15000 instances).
Wrote 1355 triples -> /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/protograph_p1.nt
Wrote 10410 triples -> /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/protograph_p2.nt
Wrote 15000 instances -> /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/entity2classes.json
Wrote 15000 instances -> /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/entity2classes_hier.json
P1 triples:  1,355
P2 triples:  10,4

In [19]:
"""Step 2: Duplicate-free random walks on P1 (scripts/_walks.py)."""
run_jrdf2vec_duplicate_free(
    P1_PATH,
    P1_WALKS_PATH,
    walks_per_entity=200,
    depth=3,
    threads=4,
    seed=42,
    token_format="angled",
    input_format="nt",
    ensure_triple_coverage=True,
)

walks = P1_WALKS_PATH.read_text(encoding="utf-8").splitlines()
walk_lengths = [len(w.split()) for w in walks]

print(f"Walk corpus: {P1_WALKS_PATH}")
print(f"Walks:       {len(walks):,}")
print(f"Tokens/walk: min={min(walk_lengths)}, max={max(walk_lengths)}, mean={sum(walk_lengths)/len(walk_lengths):.2f}")
print(f"Total tokens:{sum(walk_lengths):,}")
print("\nSample walks:")
for w in walks[:3]:
    print(" ", w[:120] + ("..." if len(w) > 120 else ""))

jRDF2Vec duplicate-free [4 threads]:   0%|          | 0/612 [00:00<?, ?entity/s]

Triple coverage: appended 40 depth-1 walks for triples dropped by per-entity trimming.
Walk corpus: /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/walks_p1.txt
Walks:       66,457
Tokens/walk: min=3, max=7, mean=6.99
Total tokens:464,463

Sample walks:
  <C_tc14_0> <P_tc14_1120> <C_tc14_544> <P_tc14_1100> <C_tc14_205> <P_tc14_407> <C_tc14_218>
  <C_tc14_0> <P_tc14_1171> <C_tc14_205> <P_tc14_839> <C_tc14_205> <P_tc14_254> <C_tc14_205>
  <C_tc14_0> <P_tc14_1157> <C_tc14_205> <P_tc14_995> <C_tc14_205> <P_tc14_363> <C_tc14_435>


In [20]:
"""Step 3: Pretrain Word2Vec on P1 protograph walks."""
pretrain_model = Word2Vec(
    sentences=LineSentence(str(P1_WALKS_PATH)),
    vector_size=200,
    window=5,
    sg=1,
    hs=0.0,
    negative=5,
    min_count=1,
    sample=0.0,
    alpha=0.025,
    min_alpha=0.0001,
    epochs=5,
    workers=4,
    seed=42,
    compute_loss=True,
)

pretrain_model.save(str(PRETRAIN_MODEL_PATH))
pretrain_model.wv.save(str(PRETRAIN_KV_PATH))
wv = pretrain_model.wv

print(f"Vocabulary:  {len(wv):,} tokens")
print(f"Vector size: {wv.vector_size}")
print(f"Saved model: {PRETRAIN_MODEL_PATH}")
print(f"Saved KV:    {PRETRAIN_KV_PATH}")

collecting all words and their counts
PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
PROGRESS: at sentence #10000, processed 69930 words, keeping 1173 word types
PROGRESS: at sentence #20000, processed 139868 words, keeping 1373 word types
PROGRESS: at sentence #30000, processed 209788 words, keeping 1506 word types
PROGRESS: at sentence #40000, processed 279660 words, keeping 1663 word types
PROGRESS: at sentence #50000, processed 349552 words, keeping 1756 word types
PROGRESS: at sentence #60000, processed 419470 words, keeping 1844 word types
collected 1967 word types from a corpus of 464463 raw words and 66457 sentences
Creating a fresh vocabulary
Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 1967 unique words (100.00% of original 1967, drops 0)', 'datetime': '2026-06-11T00:16:01.111138', 'gensim': '4.4.0', 'python': '3.13.13 (main, May 10 2026, 19:26:54) [Clang 22.1.3 ]', 'platform': 'Linux-6.1.0-49-amd64-x86_64-with-glibc2.36', 'event': 'prepa

Vocabulary:  1,967 tokens
Vector size: 200
Saved model: /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_p1.model
Saved KV:    /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_p1.kv


In [21]:
"""Step 4: Duplicate-free random walks on the TC14 instance graph."""
run_jrdf2vec_duplicate_free(
    GRAPH,
    INSTANCE_WALKS_PATH,
    walks_per_entity=200,
    depth=3,
    threads=4,
    seed=42,
    token_format="angled",
    input_format="nt",
    ensure_triple_coverage=False,
)

inst_walks = INSTANCE_WALKS_PATH.read_text(encoding="utf-8").splitlines()
inst_lengths = [len(w.split()) for w in inst_walks]

print(f"Walk corpus: {INSTANCE_WALKS_PATH}")
print(f"Walks:       {len(inst_walks):,}")
print(f"Tokens/walk: min={min(inst_lengths)}, max={max(inst_lengths)}, mean={sum(inst_lengths)/len(inst_lengths):.2f}")
print(f"Total tokens:{sum(inst_lengths):,}")

jRDF2Vec duplicate-free [4 threads]:   0%|          | 0/16924 [00:00<?, ?entity/s]

Walk corpus: /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/walks_instance.txt
Walks:       2,316,694
Tokens/walk: min=3, max=7, mean=6.96
Total tokens:16,129,710


In [22]:
"""
Step 5: Build the operative finetune model from instance walks + MASCHInE init.

Same pipeline as dbpedia.ipynb Step 6:
  1. Vocabulary from instance walks only
  2. Copy protograph rows at their native pretrained scale
  3. Initialize typed instances from most-specific class vectors (mean)
  4. Mirror initialized rows into syn1neg

Why native scale (NOT norm-matched down to gensim's random init):
gensim initializes new rows uniform in [-0.5/dim, 0.5/dim], i.e. L2 norm
~0.04, while the pretrained protograph rows have norm ~2.6. Rescaling the
pretrained rows down to 0.04 preserves their directions (epoch-0 LogReg
accuracy is still 1.0) but makes every SGNS dot product ~0: each update
then carries the maximal prediction error (sigma(0) = 0.5), and one epoch
at full LR overwrites the inherited geometry with pure instance
co-occurrence (test accuracy collapses 1.00 -> 0.65). Kept at native
scale, consistent pairs saturate the sigmoid, gradients on pretrained
directions stay small, and the protograph signal survives finetuning.
"""

FINETUNE_ALPHA = 0.005      # reduced from 0.025: finetune, don't retrain from scratch
FINETUNE_MIN_ALPHA = 0.0001


def strip_angle(x: str) -> str:
    return x[1:-1] if x.startswith("<") and x.endswith(">") else x


def init_row(model: Word2Vec, token: str, vec: np.ndarray) -> None:
    idx = model.wv.key_to_index[token]
    model.wv.vectors[idx] = vec
    model.syn1neg[idx] = vec.copy()


from _maschine_init import _class_from_extra_name


with ENTITY2CLASSES_PATH.open(encoding="utf-8") as f:
    entity2classes: dict[str, list[str]] = json.load(f)

# entity2classes keys and walk tokens are already NT tokens like "<I_tc14_0>".
# Class values are also angled ("<C_tc14_271>"); do not double-wrap with nt_term().
instance_class: dict[str, list[str]] = {}
for ent, classes in entity2classes.items():
    kept = [c for c in classes if c in wv]
    if kept:
        instance_class[ent] = kept

instance_model = Word2Vec(
    vector_size=wv.vector_size,
    window=5,
    sg=1,
    hs=0,
    negative=5,
    min_count=1,
    sample=0.0,
    alpha=FINETUNE_ALPHA,
    min_alpha=FINETUNE_MIN_ALPHA,
    workers=24,
    seed=42,
    compute_loss=True,
)
instance_model.build_vocab(
    LineSentence(str(INSTANCE_WALKS_PATH)),
    progress_per=100_000,
)
iwv = instance_model.wv
random_init_norm = float(np.linalg.norm(iwv.vectors, axis=1).mean())
pretrain_norm = float(np.linalg.norm(wv.vectors, axis=1).mean())

n_copied_props = n_copied_classes = n_inst_init = n_random = 0
init_mask = np.zeros(len(iwv), dtype=bool)
classes_used: Counter[str] = Counter()

for token in iwv.index_to_key:
    if token in wv:
        vec = wv[token].astype(np.float32, copy=True)
        init_row(instance_model, token, vec)
        init_mask[iwv.key_to_index[token]] = True
        if strip_angle(token) in p1_prop_tokens:
            n_copied_props += 1
        else:
            n_copied_classes += 1
        continue

    cls_list = instance_class.get(token)
    if not cls_list:
        extra_cls = _class_from_extra_name(strip_angle(token))
        if extra_cls is not None:
            cls_token = nt_term(extra_cls)
            if cls_token in wv:
                cls_list = [cls_token]
    if cls_list:
        vec = np.mean([wv[c] for c in cls_list], axis=0).astype(np.float32)
        init_row(instance_model, token, vec)
        init_mask[iwv.key_to_index[token]] = True
        n_inst_init += 1
        for c in cls_list:
            classes_used[c] += 1
        continue

    n_random += 1

norms = np.linalg.norm(iwv.vectors, axis=1)
freq_arr = np.array([iwv.get_vecattr(t, "count") for t in iwv.index_to_key], dtype=np.int64)
n_resources = sum(
    1
    for t in iwv.index_to_key
    if strip_angle(t).startswith("I_") or strip_angle(t).startswith("EXTRA_I_FOR_CLASS_")
)
n_init = len(iwv) - n_random

print(f"Walk corpus:     {INSTANCE_WALKS_PATH}")
print(f"  sentences:     {instance_model.corpus_count:,}")
print()
print("Entity-class mapping")
print(f"  typed, class has vector:     {len(instance_class):>9,}")
print()
print("Finetune vocabulary (walk corpus only)")
print(f"  tokens:        {len(iwv):>9,}  ({n_resources:,} instance-like tokens)")
print(f"  protograph:    {n_copied_props:,} properties, {n_copied_classes:,} classes")
print(f"  instances:     {n_inst_init:,}  ({100 * n_inst_init / max(n_resources, 1):.1f}% of instance tokens)")
print(f"  random:        {n_random:,}")
print(f"  initialized:   {100 * n_init / len(iwv):.1f}% of vocab, "
      f"{100 * freq_arr[init_mask].sum() / freq_arr.sum():.1f}% frequency-weighted")
print()
print(f"Pretrain mean norm: {pretrain_norm:.4f}  |  gensim random-init mean norm: {random_init_norm:.4f}")
print(f"Mean L2 norm — initialized: {norms[init_mask].mean():.4f}, "
      f"random: {norms[~init_mask].mean():.4f}, "
      f"ratio: {norms[init_mask].mean() / norms[~init_mask].mean():.2f}x "
      f"(intentional: large norms saturate SGNS sigmoids and protect the init)")
print()
print(f"Ready for training: instance_model ({len(iwv):,} rows, dim={iwv.vector_size})")

Word2Vec lifecycle event {'params': 'Word2Vec<vocab=0, vector_size=200, alpha=0.005>', 'datetime': '2026-06-11T00:16:12.783708', 'gensim': '4.4.0', 'python': '3.13.13 (main, May 10 2026, 19:26:54) [Clang 22.1.3 ]', 'platform': 'Linux-6.1.0-49-amd64-x86_64-with-glibc2.36', 'event': 'created'}
collecting all words and their counts
PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
PROGRESS: at sentence #100000, processed 696188 words, keeping 16618 word types
PROGRESS: at sentence #200000, processed 1392472 words, keeping 17333 word types
PROGRESS: at sentence #300000, processed 2088624 words, keeping 17578 word types
PROGRESS: at sentence #400000, processed 2784914 words, keeping 17634 word types
PROGRESS: at sentence #500000, processed 3481120 words, keeping 17661 word types
PROGRESS: at sentence #600000, processed 4177462 words, keeping 17692 word types
PROGRESS: at sentence #700000, processed 4873688 words, keeping 17705 word types
PROGRESS: at sentence #800000, proces

Walk corpus:     /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/walks_instance.txt
  sentences:     2,316,694

Entity-class mapping
  typed, class has vector:        12,037

Finetune vocabulary (walk corpus only)
  tokens:           17,914  (16,924 instance-like tokens)
  protograph:    990 properties, 0 classes
  instances:     13,966  (82.5% of instance tokens)
  random:        2,958
  initialized:   83.5% of vocab, 91.9% frequency-weighted

Pretrain mean norm: 2.5918  |  gensim random-init mean norm: 0.0408
Mean L2 norm — initialized: 2.8223, random: 0.0408, ratio: 69.16x (intentional: large norms saturate SGNS sigmoids and protect the init)

Ready for training: instance_model (17,914 rows, dim=200)


In [23]:
"""
Step 6: Finetune instance_model on instance walks; report test accuracy per epoch.

Uses the reduced finetuning LR from Step 5 (0.005 -> 0.0001). With the
from-scratch LR (0.025) the protograph init decays 1.00 -> 0.79 over 5
epochs even at native scale; at 0.005 it holds ~0.99. (With the old
norm-matched-down init it collapsed to 0.65 regardless of LR.)
"""
import logging

from gensim.models.callbacks import CallbackAny2Vec
from sklearn.metrics import accuracy_score

from _evaluate import load_labeled_txt, make_classifiers, tokens_to_embeddings

logging.basicConfig(format="%(message)s", level=logging.INFO)


def test_logreg_accuracy(model: Word2Vec) -> float:
    emb = np.asarray(model.wv.vectors, dtype=np.float32)
    word2idx = dict(model.wv.key_to_index)
    train_tokens, y_train = load_labeled_txt(TRAIN_TXT)
    test_tokens, y_test = load_labeled_txt(TEST_TXT)
    x_train, _ = tokens_to_embeddings(
        train_tokens, emb, word2idx, "Embed train entities", 2048, progress=False
    )
    x_test, _ = tokens_to_embeddings(
        test_tokens, emb, word2idx, "Embed test entities", 2048, progress=False
    )
    clf = make_classifiers(max_iter=1000, seed=42)["LogReg"]
    clf.fit(x_train, y_train)
    return float(accuracy_score(y_test, clf.predict(x_test)))


rows = [(0, test_logreg_accuracy(instance_model))]


class EpochAccuracyCallback(CallbackAny2Vec):
    def on_epoch_end(self, model):
        rows.append((len(rows), test_logreg_accuracy(model)))


instance_model.train(
    corpus_iterable=LineSentence(str(INSTANCE_WALKS_PATH)),
    total_examples=instance_model.corpus_count,
    epochs=5,
    start_alpha=FINETUNE_ALPHA,
    end_alpha=FINETUNE_MIN_ALPHA,
    report_delay=1.0,
    callbacks=[EpochAccuracyCallback()],
)

instance_model.save(str(INSTANCE_MODEL_PATH))
instance_model.wv.save(str(INSTANCE_KV_PATH))

print(f"Saved model: {INSTANCE_MODEL_PATH}")
print(f"Saved KV:    {INSTANCE_KV_PATH}")
print()
print(f"{'epoch':>5}  {'accuracy':>8}")
print("-" * 16)
for epoch, accuracy in rows:
    print(f"{epoch:>5}  {accuracy:>8.4f}")

Word2Vec lifecycle event {'msg': 'training model with 24 workers on 17914 vocabulary and 200 features, using sg=1 hs=0 sample=0.0 negative=5 window=5 shrink_windows=True', 'datetime': '2026-06-11T00:16:15.868222', 'gensim': '4.4.0', 'python': '3.13.13 (main, May 10 2026, 19:26:54) [Clang 22.1.3 ]', 'platform': 'Linux-6.1.0-49-amd64-x86_64-with-glibc2.36', 'event': 'train'}
EPOCH 0 - PROGRESS: at 11.40% examples, 1660422 words/s, in_qsize 30, out_qsize 2
EPOCH 0 - PROGRESS: at 24.42% examples, 1828268 words/s, in_qsize 47, out_qsize 0
EPOCH 0 - PROGRESS: at 36.44% examples, 1847739 words/s, in_qsize 48, out_qsize 0
EPOCH 0 - PROGRESS: at 47.29% examples, 1818166 words/s, in_qsize 38, out_qsize 1
EPOCH 0 - PROGRESS: at 58.32% examples, 1804642 words/s, in_qsize 48, out_qsize 1
EPOCH 0 - PROGRESS: at 70.78% examples, 1837122 words/s, in_qsize 41, out_qsize 3
EPOCH 0 - PROGRESS: at 81.25% examples, 1816109 words/s, in_qsize 47, out_qsize 0
EPOCH 0 - PROGRESS: at 92.35% examples, 1809472 wo

Saved model: /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_instance.model
Saved KV:    /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_instance.kv

epoch  accuracy
----------------
    0    1.0000
    1    0.9994
    2    0.9944
    3    0.9888
    4    0.9856
    5    0.9844


In [24]:
"""Step 7: Evaluate finetuned embeddings on TC14 test split."""
from _evaluate import format_eval_metrics_lines, run_evaluation

eval_res = run_evaluation(
    TEST_TXT,
    INSTANCE_MODEL_PATH,
    train_path=TRAIN_TXT,
)
print("\n".join(format_eval_metrics_lines(eval_res)))

metrics_path = OUT_DIR / "eval_metrics.txt"
metrics_path.write_text("\n".join(format_eval_metrics_lines(eval_res)) + "\n", encoding="utf-8")
print(f"Wrote {metrics_path}")

loading Word2Vec object from /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_instance.model
loading wv recursively from /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_instance.model.wv.* with mmap=None
setting ignored attribute cum_table to None
Word2Vec lifecycle event {'fname': '/home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/word2vec_instance.model', 'datetime': '2026-06-11T00:17:00.312716', 'gensim': '4.4.0', 'python': '3.13.13 (main, May 10 2026, 19:26:54) [Clang 22.1.3 ]', 'platform': 'Linux-6.1.0-49-amd64-x86_64-with-glibc2.36', 'event': 'loaded'}


Embed train entities |          | 0/1 chunks [00:00<?, ?chunk/s]

Embed test entities |          | 0/1 chunks [00:00<?, ?chunk/s]

Test metrics (binary classification, positive class = 1)
────────────────────────────────────────────────────

LogReg
  accuracy      0.9844
  precision     0.9850
  recall        0.9838
  f1            0.9844

────────────────────────────────────────────────────
Wrote /home/bnukhkadiev/Knowledge-Graphs/notebooks/synthetic_tc14/eval_metrics.txt
